# V6.5-14 — YOLO26n + AdamW + ReduceLROnPlateau
Cùng công thức với v6p5-13 (AdamW lr0=0.001, wd=0.001, Plateau, batch 32, seed, dataset), chỉ đổi kiến trúc **yolo11n → yolo26n** để đo riêng ảnh hưởng của model.
Mục tiêu triển khai: Raspberry Pi 3/5 chỉ CPU, không real-time (phát hiện lỗi → gửi cảnh báo). YOLO26n NMS-free, bỏ DFL nên export NCNN/TFLite gọn và nhanh hơn trên CPU.
Chạy CONFIG → GPU GATE → SCHEDULER → TRAIN → FAIR EVAL. Không chạy song song với run v6p5-13 (tranh GPU/CPU).
Early stop theo **mAP50–95 trên VAL**. Scheduler giảm LR sau 3 epoch không cải thiện, early stop sau 7 epoch.


In [2]:
# CONFIG + restore local artifacts, không tải lại dataset
from pathlib import Path
import json, shutil, sys
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "Uav-Powerline-Defect-Detection"]
PROJECT = next(p for p in candidates if (p / "pyproject.toml").is_file() and (p / "legacy/notebooks/v6p5").is_dir())
DATASET = PROJECT.parent / "datasets/v6p5_roboflow_ablation"
DATA_YAML = DATASET / "data.yaml"
MODEL = PROJECT / "legacy/notebooks/v6p5/weights/yolo26n.pt"  # YOLO26n: NMS-free, không DFL, tối ưu cho CPU/edge (Raspberry Pi)
PREV_BEST = PROJECT / "artifacts/v6p5_train_local/20261009T024019Z_roboflow/runs/yolo11n_v6p5_roboflow_ablation_640/weights/best.pt"
# Baseline cùng công thức (AdamW + Plateau, batch 32) nhưng là YOLO11n — chỉ khác kiến trúc
Y11N_PLATEAU_BEST = PROJECT / "artifacts/v6p5_train_local/20261010T060816Z_plateau_wd001/runs/yolo11n_v6p5_plateau_wd001/weights/best.pt"
from datetime import datetime, timezone
EXPERIMENT = "plateau_wd001"  # YOLO26 không có DFL -> các biến thể dfl_* không có tác dụng
VARIANTS = {
    "plateau_wd001": {"box": 7.5, "dfl": 1.5},
    "plateau_box_plus1": {"box": 8.5, "dfl": 1.5},
    "plateau_box_plus2": {"box": 9.5, "dfl": 1.5},
    "plateau_dfl_plus025": {"box": 7.5, "dfl": 1.75},
    "plateau_dfl_plus05": {"box": 7.5, "dfl": 2.0},
}
LOSS = VARIANTS[EXPERIMENT]
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_y26n_" + EXPERIMENT
RUN_NAME = "yolo26n_v6p5_" + EXPERIMENT
ROOT = PROJECT / "artifacts/v6p5_train_local" / RUN_ID
RUN_DIR = ROOT / "runs" / RUN_NAME
NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand"]
IMGSZ, EPOCHS, BATCH, DEVICE, WORKERS, SEED = 640, 100, 32, 0, 8, 20261007  # batch=32 + nbs=64 -> effective batch vẫn 64; workers=8 song song dataloader
for required in [DATA_YAML, MODEL, PREV_BEST]:
    if not required.is_file(): raise FileNotFoundError(required)
print("New experiment:", EXPERIMENT, LOSS)
print("Output:", RUN_DIR)


New experiment: plateau_wd001 {'box': 7.5, 'dfl': 1.5}
Output: /teamspace/studios/this_studio/Uav-Powerline-Defect-Detection/artifacts/v6p5_train_local/20261010T074752Z_y26n_plateau_wd001/runs/yolo26n_v6p5_plateau_wd001


In [3]:
# 1. GPU + DATASET GATE (NO TRAINING)
import yaml
import torch
import ultralytics

if not torch.cuda.is_available():
    raise RuntimeError("CUDA chưa khả dụng trong kernel WSL.")
if not 0 <= DEVICE < torch.cuda.device_count():
    raise ValueError(f"DEVICE={DEVICE} không hợp lệ.")
if ultralytics.__version__ != "8.4.172":
    raise RuntimeError(
        f"Cần ultralytics==8.4.172 để so sánh công bằng; hiện tại={ultralytics.__version__}"
    )
cfg = yaml.safe_load(DATA_YAML.read_text())
cfg_names = cfg.get("names", {})
dataset_names = ([cfg_names[k] for k in sorted(cfg_names, key=int)]
                 if isinstance(cfg_names, dict) else list(cfg_names))
if dataset_names != NAMES:
    raise ValueError(f"Taxonomy mismatch: {dataset_names}")
dataset_root = Path(cfg["path"])
for split in ["train", "val"]:
    image_dir = dataset_root / cfg[split]
    if not image_dir.is_dir():
        raise FileNotFoundError(image_dir)
    print(split, len([p for p in image_dir.iterdir() if p.is_file()]))
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__, "| Ultralytics:", ultralytics.__version__)



train 18334
val 2272
GPU: NVIDIA L4
PyTorch: 2.14.0+cu130 | Ultralytics: 8.4.172


In [4]:
# Custom scheduler — Ultralytics gọi step() đầu epoch, Plateau cần metric cuối epoch.
from torch.optim.lr_scheduler import ReduceLROnPlateau
from ultralytics.models.yolo.detect import DetectionTrainer

class MetricPlateau(ReduceLROnPlateau):
    def step(self, metrics=None, epoch=None):
        if metrics is not None:
            return super().step(metrics, epoch)

class PlateauTrainer(DetectionTrainer):
    def _setup_scheduler(self):
        # Warmup kết thúc tại lr0, không có linear/cosine chạy song song.
        self.lf = lambda epoch: 1.0
        # Warmup của Ultralytics đọc "initial_lr"; LambdaLR tự gán, ReduceLROnPlateau thì không.
        for group in self.optimizer.param_groups:
            group.setdefault("initial_lr", group["lr"])
        self.scheduler = MetricPlateau(
            self.optimizer, mode="max", factor=0.5, patience=2,
            threshold=0.0, threshold_mode="abs", min_lr=1e-6,
        )
        if not getattr(self, "_plateau_callback_added", False):
            self.add_callback("on_fit_epoch_end", self._step_plateau)
            self._plateau_callback_added = True

    @staticmethod
    def _step_plateau(trainer):
        if trainer.epoch + 1 <= trainer.args.warmup_epochs:
            return
        metric = float(trainer.metrics["metrics/mAP50-95(B)"])
        trainer.scheduler.step(metric)
        print(f"Plateau monitor mAP50-95={metric:.5f}; next LR={trainer.optimizer.param_groups[0]['lr']:.6g}")

# Quick check: đầu epoch không đổi LR; 3 lần không tăng giảm LR một nửa.
_probe_parameter = torch.nn.Parameter(torch.zeros(1))
_probe_optimizer = torch.optim.AdamW([_probe_parameter], lr=0.001)
_probe_scheduler = MetricPlateau(_probe_optimizer, mode="max", factor=0.5, patience=2)
_probe_scheduler.step()
assert _probe_optimizer.param_groups[0]["lr"] == 0.001
for value in [0.4, 0.4, 0.4, 0.4]:
    _probe_scheduler.step(value)
assert _probe_optimizer.param_groups[0]["lr"] == 0.0005
print("Plateau scheduler check passed")


Plateau scheduler check passed


In [ ]:
# TRAIN NEW EXPERIMENT — không resume checkpoint cosine với scheduler mới
from ultralytics import YOLO
model = YOLO(str(MODEL))
model.train(
    trainer=PlateauTrainer,
    data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH,
    device=DEVICE, workers=WORKERS,
    optimizer="AdamW", lr0=0.001, weight_decay=0.001,
    cos_lr=False, warmup_epochs=3.0, patience=7,
    box=LOSS["box"], dfl=LOSS["dfl"], close_mosaic=10,
    pretrained=True, seed=SEED, deterministic=True,
    cache=False, save=True, save_period=5,
    project=str(ROOT / "runs"), name=RUN_NAME,
    exist_ok=False, plots=False, verbose=True,
)
best = RUN_DIR / "weights/best.pt"
if not best.is_file(): raise FileNotFoundError(best)
print("Best:", best)


New https://pypi.org/project/ultralytics/8.4.175 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.172 🚀 Python-3.14.7 torch-2.14.0+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/teamspace/studios/this_studio/datasets/v6p5_roboflow_ablation/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mix

In [ ]:
# 3. FAIR EVAL — YOLO26N VS YOLO11N (CÙNG CÔNG THỨC) VS MUSGD TRÊN CÙNG VAL
import gc
import pandas as pd

try:
    model.cpu()
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

rows = []
candidates = [("y26n_adamw_plateau", best), ("y11n_adamw_plateau", Y11N_PLATEAU_BEST), ("y11n_musgd_auto", PREV_BEST)]
for tag, weights in [(t, w) for t, w in candidates if Path(w).is_file()]:
    metrics = YOLO(str(weights)).val(
        data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=BATCH,
        device=DEVICE, workers=WORKERS, plots=False, verbose=False,
        project=str(ROOT / "eval"), name=tag, exist_ok=True,
    ).box
    rows.append({"model": tag, "class": "ALL",
                 "AP50": float(metrics.map50), "AP50_95": float(metrics.map)})
    for cid, ap50, ap in zip(metrics.ap_class_index, metrics.ap50, metrics.ap):
        rows.append({"model": tag, "class": NAMES[int(cid)],
                     "AP50": float(ap50), "AP50_95": float(ap)})
result = pd.DataFrame(rows)
result.to_csv(ROOT / "y26n_vs_y11n_eval.csv", index=False)
display(result.pivot_table(index="class", columns="model",
                           values=["AP50", "AP50_95"]).round(4))


So sánh chính: `y26n_adamw_plateau` vs `y11n_adamw_plateau` (cùng công thức). `y11n_musgd_auto` giữ làm mốc tham chiếu.
YOLO26 không có DFL (reg_max=1) nên tham số `dfl` không có tác dụng; chỉ biến thể box_* còn ý nghĩa.
Checkpoint của custom scheduler này chưa hỗ trợ resume đầy đủ trạng thái Plateau. Nếu kernel ngắt, giữ artifact; không gọi resume=True rồi coi là tiếp nối chính xác scheduler.
